[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C61_Detection_Practice_Interview_Course/04_narrative/04_project_narrative.ipynb)

# 04 · 项目叙事（STAR 检测版 / 指标配对 / 追问覆盖度 / 完整 TSR 样例）

目标：把「讲项目」从一门玄学变成一个**可以被程序检查的结构**。
本 notebook 不需要 numpy 之外的任何依赖，全部是纯标准库的规则检查器 ——
因为叙事的问题从来不是算法问题，是**信息完整性**问题。

本 notebook 你会亲手实现：
1. **叙事的数据结构**：把 STAR 四格拆成 15 个必填字段
2. **完整性自检器**：一眼看出你的故事缺哪一格
3. **指标配对检查器**：报了收益却没报代价的地方全部标红
4. **归因与验证检查器**：机器区分「做了什么」和「为什么这么做」
5. **一个完整的 TSR 项目叙事样例**（满分，可直接改写自用）
6. **三层追问覆盖度检查**：口径 / 方差 / 机制 —— 你在哪一层会崩
7. **10 分钟时间预算器**：A 段是不是被你讲短了
8. **简历 bullet 打分器**：六项检查，机器先筛一遍

> 心智模型：**面试官看不到你做过什么，只能看到你说的话。
> 叙事准备不是练口才，是把当初没记下来的信息重新捡回来。**

## 1 · 把叙事变成数据结构

STAR 四格，15 个必填字段。**字段缺失 = 面试官必然追问的地方**。
被追问本身不可怕，但它意味着「这个维度你没有主动覆盖」——
而多数评分表里「未观察到」的处理方式和「不合格」是一样的。

In [ ]:
from collections import OrderedDict

# STAR 的检测版 schema：每一格的必填字段，以及它对应的信息类型
STORY_SCHEMA = OrderedDict([
    ('S', OrderedDict([
        ('scene',           '什么任务 / 什么数据 / 什么部署环境'),
        ('baseline',        'baseline 指标（必须有数字）'),
        ('metric_protocol', '评测口径（哪个 mAP 定义 / 工作点 / 延迟含不含 NMS）'),
        ('how_found',       '问题是怎么被发现的（线上 badcase？分桶评测？门禁？）'),
    ])),
    ('T', OrderedDict([
        ('goal',        '目标：哪个指标从多少提到多少'),
        ('constraints', '约束：延迟 / 显存 / 算力平台 / 标注预算 / 排期'),
        ('frozen',      '不能动的东西：下游接口 / 旧场景不掉点 / 工具链'),
    ])),
    ('A', OrderedDict([
        ('why',               '归因链：从现象到假设到方案（**最高区分度**）'),
        ('actions',           '做了什么：方法名 + 关键超参 + 数据量级'),
        ('failed',            '试过什么没成功（含归因）'),
        ('validation_design', '怎么保证结论可信：种子 / 消融 / 切片'),
    ])),
    ('R', OrderedDict([
        ('before_after',    'baseline -> 改动后（同口径，成对给数）'),
        ('costs',           '代价：延迟 / 显存 / 其他场景 / 工程复杂度'),
        ('verification',    '验证方式：切片 / 门禁 / 多种子 / 灰度'),
        ('business_impact', '业务影响（检出距离 / 反应时间 / 投诉率…）'),
    ])),
])

N_FIELDS = sum(len(v) for v in STORY_SCHEMA.values())
print('STAR 四格共 %d 个必填字段' % N_FIELDS)
for g, fields in STORY_SCHEMA.items():
    print('  [%s] %s' % (g, ' / '.join(fields)))
assert N_FIELDS == 15
assert list(STORY_SCHEMA) == ['S', 'T', 'A', 'R']
print('\n✅ schema 就位。注意 A 格有 4 个字段，其中 why 与 failed 是多数人完全空着的两个。')

## 2 · 完整性自检器：你的故事缺哪一格

规则很粗暴：字段值为空字符串 / 空列表 / None 一律视为缺失。
先拿一个「弱版故事」开刀 —— 这是多数人第一次讲项目的真实样子。

In [ ]:
def is_filled(v):
    """非空字符串 / 非空容器视为已填。"""
    if v is None:
        return False
    if isinstance(v, str):
        return len(v.strip()) > 0
    if isinstance(v, (list, tuple, dict, set)):
        return len(v) > 0
    return True

def check_completeness(story):
    """返回 (已填数, 总数, 缺失字段列表[(格子, 字段名)])。"""
    missing, filled = [], 0
    for g, fields in STORY_SCHEMA.items():
        blk = story.get(g, {})
        for f in fields:
            if is_filled(blk.get(f)):
                filled += 1
            else:
                missing.append((g, f))
    return filled, N_FIELDS, missing

WEAK_STORY = {
    'S': {'scene': '交通标志检测项目'},
    'T': {'goal': '提升 mAP'},
    'A': {'actions': ['加了数据增强', '换了更大的模型', '调了超参']},
    'R': {'before_after': [{'kind': 'ap', 'name': 'mAP', 'before': 0.58, 'after': 0.615}]},
}

f, n, miss = check_completeness(WEAK_STORY)
print('弱版故事完整度：%d/%d = %.0f%%' % (f, n, 100.0 * f / n))
print('缺失字段（每一条都是一次必然的追问）：')
for g, fld in miss:
    print('   [%s] %-18s <- %s' % (g, fld, STORY_SCHEMA[g][fld]))
assert (f, n) == (4, 15)
for key in [('A', 'why'), ('A', 'failed'), ('R', 'costs'), ('S', 'metric_protocol')]:
    assert key in miss, key
print('\n✅ 弱版只覆盖 27% 的必填信息，其余 11 项全靠面试官追问补齐 —— 45 分钟不够用。')

## 3 · 指标必须成对：收益 → 代价的配对检查

工程里没有免费的提升。**报了收益却报不出代价，只有两种可能：没测，或者不敢说。**
下面把「哪种收益必须配哪种代价」写成规则表，让机器替你先查一遍。

In [ ]:
# 收益类型 -> 必须给出的代价类型（给出其中任意一项即算配对成功）
PAIR_RULES = OrderedDict([
    ('ap',           ['latency', 'memory', 'train_cost']),
    ('recall',       ['fp_rate']),
    ('small_bucket', ['large_bucket', 'latency']),
    ('rare_class',   ['head_class', 'overall_ap']),
    ('new_scene',    ['old_scene']),
    ('latency_down', ['accuracy']),
    ('nms_free',     ['train_cost', 'query_cap']),
])

COST_DESC = {
    'latency': '端到端延迟', 'memory': '显存峰值', 'train_cost': '训练成本(GPU-h)',
    'fp_rate': '同工作点的 FP 率', 'large_bucket': '大目标桶', 'head_class': 'head 类别',
    'overall_ap': '总体 AP', 'old_scene': '旧场景切片', 'accuracy': '精度掉点',
    'query_cap': 'query 数上限',
}

def check_metric_pairs(story):
    """返回 [(收益类型, 可接受的代价候选)]；空列表 = 全部配对齐了。"""
    gains = story.get('R', {}).get('before_after', []) or []
    costs = story.get('R', {}).get('costs', []) or []
    have = set(c.get('kind') for c in costs)
    warn = []
    for g in gains:
        need = PAIR_RULES.get(g.get('kind'))
        if need and not (have & set(need)):
            warn.append((g.get('kind'), need))
    return warn

w = check_metric_pairs(WEAK_STORY)
print('弱版故事的配对告警：')
for k, need in w:
    print('   收益 %-13s 缺代价，至少要给一项：%s'
          % (k, ' / '.join(COST_DESC[c] for c in need)))
assert w == [('ap', ['latency', 'memory', 'train_cost'])], w
print('\n✅ 只报「mAP 0.58 -> 0.615」而不报延迟，是最经典的被追问点。')
print('   顺带记住这句可以直接说的话：「这个改动我记的是一对数：小目标桶 +0.17，延迟 +1.7 ms。」')

## 4 · 归因与验证：机器区分「做了什么」和「为什么这么做」

「做了什么」是工序，可以照抄；**「为什么这么做」是决策函数，抄不走** ——
而面试官招的正是这个决策函数。用关键词指纹做一次粗筛。

In [ ]:
import re

# 归因证据的指纹：真做过误差归因的人，话里几乎一定出现这类词
EVIDENCE_MARKERS = ['TIDE', '分桶', '混淆矩阵', 'PR 曲线', '误差分解', '占', '排除', '归因']
# 验证纪律的指纹
SEED_PAT      = re.compile(r'([0-9]+)\s*(个)?\s*(seed|种子)', re.I)
TEST_MARKERS  = ['配对', 'bootstrap', 't 检验', '显著', 'p<', 'p <']
SLICE_MARKERS = ['切片', '分桶', '门禁', '回归', '灰度']

def check_attribution(story):
    """A.why 必须存在且含归因证据指纹。返回 (ok, 命中的指纹列表)。"""
    why = story.get('A', {}).get('why', '') or ''
    hits = [m for m in EVIDENCE_MARKERS if m in why]
    return (len(why.strip()) > 0 and len(hits) > 0), hits

def check_validation(story):
    """返回 {种子数, 是否有显著性检验, 是否有切片/门禁}。"""
    text = (story.get('A', {}).get('validation_design', '') or '') + ' ' + \
           (story.get('R', {}).get('verification', '') or '')
    m = SEED_PAT.search(text)
    return {'n_seeds': int(m.group(1)) if m else 0,
            'has_test': any(t in text for t in TEST_MARKERS),
            'has_slice': any(s in text for s in SLICE_MARKERS)}

ok, hits = check_attribution(WEAK_STORY)
v = check_validation(WEAK_STORY)
print('弱版：归因 ok =', ok, '  命中指纹 =', hits)
print('弱版：验证 =', v)
assert ok is False and hits == []
assert v == {'n_seeds': 0, 'has_test': False, 'has_slice': False}
print('\n✅ 单种子 + 无检验 + 无切片 = 「+0.3 是不是噪声」这一刀必然被砍中（见 C61 m01）。')

## 5 · 综合自检器：一次跑完四项检查

输出一个 0–100 的分数与一份告警清单。
**分数不重要，告警清单才是产出物 —— 每一条告警都是一次你没准备的追问。**

In [ ]:
def narrative_audit(story, name='(未命名)'):
    f, n, miss = check_completeness(story)
    pairs = check_metric_pairs(story)
    attr_ok, _ = check_attribution(story)
    val = check_validation(story)

    warnings = []
    for g, fld in miss:
        warnings.append('[缺字段] %s.%s —— %s' % (g, fld, STORY_SCHEMA[g][fld]))
    for k, need in pairs:
        warnings.append('[缺代价] 报了 %s 的收益，却没给 %s 里的任何一项' % (k, '/'.join(need)))
    if not attr_ok:
        warnings.append('[缺归因] A.why 里没有归因证据（TIDE / 分桶 / 混淆矩阵 / 排除…）')
    if val['n_seeds'] < 3:
        warnings.append('[缺验证] 种子数 %d < 3，单种子结论不该进决策' % val['n_seeds'])
    if not val['has_test']:
        warnings.append('[缺验证] 没有显著性检验（配对 / bootstrap）')
    if not val['has_slice']:
        warnings.append('[缺验证] 没有分场景切片或回归门禁')

    score = 60.0 * f / n                                  # 完整度 60 分
    score += 15.0 if not pairs else 0.0                   # 指标配对 15 分
    score += 10.0 if attr_ok else 0.0                     # 归因 10 分
    score += 5.0 * ((val['n_seeds'] >= 3) + val['has_test'] + val['has_slice'])  # 验证 15 分
    return {'name': name, 'score': round(score, 1), 'filled': (f, n), 'warnings': warnings}

rep = narrative_audit(WEAK_STORY, '弱版故事')
print('%s  得分 %.1f/100  完整度 %d/%d' % (rep['name'], rep['score'], rep['filled'][0], rep['filled'][1]))
print('告警 %d 条：' % len(rep['warnings']))
for line in rep['warnings']:
    print('   -', line)
assert rep['score'] < 25
assert len(rep['warnings']) == 16, len(rep['warnings'])
print('\n✅ 16 条告警 = 16 次追问。这就是「讲不清项目」的解剖学解释。')

## 6 · 一个完整的 TSR 项目叙事样例（满分，可直接改写自用）

**用法**：把里面的数字和方法换成你自己的，保持结构不变。
如果某一格你换不出内容 —— 那正是你要补的功课，而不是要绕过的格子。

In [ ]:
TSR_STORY = {
 'S': {
   'scene': 'TT100K 交通标志检测，目标平台是车端 SoC；单帧感知预算 12 ms（TSR 只是共享算力上的一个任务）',
   'baseline': 'YOLOv8s @640，COCO 口径 mAP50-95 = 0.58；<32px 桶 recall@FP=0.1/帧 = 0.44',
   'metric_protocol': 'mAP50-95（COCO 口径）+ 按标志像素尺寸分四桶（<32 / 32-64 / 64-128 / >128）的 recall，工作点固定在 FP=0.1 帧；延迟口径 = batch1、含预处理与 NMS、p50 与 p99 都报',
   'how_found': '分桶评测发现的：总 mAP 看不出问题，把召回按像素尺寸切开后 <32px 桶只有 0.44，而它占了全部漏检实例的 71%',
 },
 'T': {
   'goal': '<32px 桶 recall 从 0.44 提到 0.60（换算成业务：首次检出距离从 42m 提前到 55m 以上）',
   'constraints': '端到端 p99 <= 12 ms；显存 <= 1.2 GB；工具链不允许自定义 plugin；标注预算 5k 框',
   'frozen': '下游接口不变（类别表与置信度字段）；其他尺寸桶掉点不得超过 0.5（回归门禁）',
 },
 'A': {
   'why': 'TIDE 误差分解显示 Miss 占 mAP 损失的 62%，其中 89% 落在 <32px 桶，Cls 与 Loc 合计只占 21%。据此排除了「换更强 backbone」这条路（它主要改善 Cls/Loc，而我的损失不在那）。再用针孔模型核算物理量：px = f*S/Z，60m 外 60cm 的限速牌在 1920x1080 / 60 度 FOV 上约 17px，缩到 640 输入只剩 5.7px —— 在 stride 8 的 P3 上连一个格子都占不满。问题的根在分辨率与层级，不在特征表达能力',
   'actions': [
     '输入分辨率 640 -> 960：最直接的杠杆，先做它是为了量出天花板',
     '小尺度 copy-paste：从标注抠出 4100 个稀有类与小尺寸实例建库；贴入尺度按透视约束采样（贴入位置的 y 坐标决定允许的尺度区间），位置限制在路侧可行区域',
     'Mosaic 保留，但最后 10 个 epoch 关闭（close-mosaic），避免拼接分布与真实推理分布不符',
     '评测侧固化分桶脚本与工作点，进 CI 做回归门禁',
   ],
   'failed': [
     '加 P2 层（stride 4）：<32px 桶 0.44 -> 0.55，是所有方案里效果最好的；但延迟 6.1 -> 9.4 ms、显存 +40%，中配 SoC 放不下，放弃。保留为「天花板参考」：它说明小目标还有 0.11 的空间，剩下的工作是找便宜的方式逼近它',
     '第一版 copy-paste 完全没效果（+0.01）。查了两天发现贴入的框在后续仿射增强里没有跟随变换，图像和标注对不上。修完之后才有 +0.06。此后把「新增数据管线算子必须先过可视化 + 框面积分布对比」固化成流程',
     '试过 scale-aware 损失加权（小目标 x2）：小目标桶 +0.02、大目标桶 -0.04，总 mAP 净负。判定为「沿曲线移动」而不是「推动曲线」，放弃',
   ],
   'validation_design': '每个配置跑 3 seeds；主指标用配对 bootstrap（重采样单位是图像）算置信区间；消融用减法式（从完整配方里逐个去掉），同 epoch、同调参预算；同时看四个尺寸桶 + 四个场景切片（白天 / 夜间 / 雨天 / 隧道口）',
 },
 'R': {
   'before_after': [
     {'kind': 'small_bucket', 'name': '<32px 桶 recall@FP=0.1', 'before': 0.44, 'after': 0.61},
     {'kind': 'ap',           'name': 'mAP50-95',               'before': 0.58, 'after': 0.615},
     {'kind': 'new_scene',    'name': '夜间切片 recall',        'before': 0.31, 'after': 0.40},
   ],
   'costs': [
     {'kind': 'latency',      'name': '端到端 p50 延迟(ms)',    'before': 6.1,  'after': 7.8},
     {'kind': 'memory',       'name': '显存峰值(GB)',           'before': 0.9,  'after': 1.2},
     {'kind': 'train_cost',   'name': '训练时长(GPU-h)',        'before': 20.0, 'after': 28.0},
     {'kind': 'large_bucket', 'name': '>128px 桶 recall',       'before': 0.93, 'after': 0.927},
     {'kind': 'fp_rate',      'name': 'FP/帧（固定工作点）',    'before': 0.10, 'after': 0.10},
     {'kind': 'old_scene',    'name': '白天切片 recall',        'before': 0.79, 'after': 0.79},
   ],
   'verification': '3 seeds 的配对 bootstrap：<32px 桶提升 p<0.01；四个尺寸桶 + 四个场景切片全部过回归门禁（容差 0.5）；灰度阶段用影子模式跑了两周的真实回传数据',
   'business_impact': '首次检出距离 42m -> 58m —— 按 80 km/h 计，给下游多出约 0.7 秒反应时间；限速牌相关误报投诉未上升（FP/帧 工作点未变）',
 },
}

rep2 = narrative_audit(TSR_STORY, 'TSR 完整叙事')
print('%s  得分 %.1f/100  完整度 %d/%d'
      % (rep2['name'], rep2['score'], rep2['filled'][0], rep2['filled'][1]))
print('告警：', rep2['warnings'] if rep2['warnings'] else '无 ✅')
assert rep2['filled'] == (15, 15)
assert rep2['warnings'] == [], rep2['warnings']
assert rep2['score'] == 100.0
print('\n对比：弱版 %.1f 分 / %d 条告警   ->   完整版 %.1f 分 / 0 条告警'
      % (rep['score'], len(rep['warnings']), rep2['score']))
print('注意：这两个故事描述的是**同一个项目**。区别只在于当初有没有把信息记下来。')

## 7 · 三层追问覆盖度：你在哪一层会崩

对叙事里的**每一个 claim**，做机械的三层展开：

1. **口径层** —— 这是怎么测的（baseline / 定义 / 工作点 / 是不是单变量）
2. **方差层** —— 这个数可信吗（几个种子 / 显著吗 / 副作用 / 有没有泄漏）
3. **机制层** —— 为什么有效，**以及什么时候会无效**

经验值：多数人在第 2 层的「方差」和第 3 层的「何时无效」上崩，
而这两处恰好是区分资深度最有效的地方。

In [ ]:
LAYERS = ('口径', '方差', '机制')

CLAIMS = [
 {'text': 'copy-paste 让稀有类 AP +5',
  '口径': '稀有类定义为训练集实例数 <200 的 47 类；AP50-95；baseline 是同配置只关掉 copy-paste（严格单变量）',
  '方差': '稀有类验证实例只有 1.2k，AP 方差大 -> 跑了 5 seeds，均值 +4.6、std 1.1，配对 bootstrap p<0.01；head 类 -0.2',
  '机制': '正样本数直接上升 + 背景多样性增加，缓解「稀有类总出现在同一种背景」的偏差。无效甚至有害的情形：贴入尺度不符合透视、接缝成为捷径特征、遮挡了原有目标但标注未更新、贴到不可能出现的位置'},
 {'text': '输入 640->960 让 <32px 桶 +0.11',
  '口径': '同 epoch、同增强、同评测集；延迟口径含预处理与 NMS，batch=1',
  '方差': '3 seeds，std 0.02，提升远大于噪声',
  '机制': ''},
 {'text': 'close-mosaic 最后 10 epoch 有效',
  '口径': '在 960 配置上做的减法式消融',
  '方差': '',
  '机制': ''},
 {'text': 'P2 层是天花板参考（+0.11 但放不下）',
  '口径': '同配置只加 P2 头；延迟在同一块卡、同一 batch 下测',
  '方差': '2 seeds（因为它本来就不打算上线，只用来估上界）',
  '机制': 'stride 4 的特征图上 8px 目标占 2x2 格，信息没有被下采样抹掉；代价是 token 数按分辨率平方增长'},
]

def layer_coverage(claims):
    """返回 (每层已填数 dict, 总已填, 总格子数)。"""
    per = OrderedDict((L, sum(1 for c in claims if is_filled(c.get(L)))) for L in LAYERS)
    return per, sum(per.values()), len(claims) * len(LAYERS)

per, tot, cells = layer_coverage(CLAIMS)
print('三层追问覆盖度：')
for L in LAYERS:
    print('   %-4s %d/%d = %3.0f%%' % (L, per[L], len(CLAIMS), 100.0 * per[L] / len(CLAIMS)))
print('   ----  总计 %d/%d = %.0f%%' % (tot, cells, 100.0 * tot / cells))
assert per['口径'] == 4 and per['方差'] == 3 and per['机制'] == 2
assert (tot, cells) == (9, 12)
print('\n✅ 覆盖率 75%，短板在「机制」层 —— 和经验规律一致。')
print('   目标：面试前把总覆盖率打到 80% 以上，且「机制」层不低于 75%。')

## 8 · 10 分钟故事的时间预算

多数人的实际分配是「S 讲三分钟、A 讲两分钟」—— 因为背景好讲、判断难讲。
**A 段必须占一半以上时间**，它是全部区分度所在。用秒表卡一次，你会吓一跳。

In [ ]:
CHARS_PER_MIN = 220        # 中文口语的舒适语速（含停顿）

PLAN_10MIN = [('S 场景与口径', 75), ('T 目标与约束', 60), ('A 归因', 75),
              ('A 方案与取舍', 180), ('A 验证设计', 60), ('R 结果与代价', 90),
              ('留钩子', 60)]

BAD_PLAN = [('S 场景', 180), ('T 目标', 60), ('A 方案', 120), ('R 结果', 60)]

def audit_time_plan(plan, total=600, min_a_share=0.50):
    s = sum(t for _, t in plan)
    a = sum(t for name, t in plan if name.startswith('A'))
    return {'total_s': s, 'a_share': a / s, 'fits': abs(s - total) <= 30,
            'a_ok': a / s >= min_a_share, 'words': int(s / 60.0 * CHARS_PER_MIN)}

for label, plan in [('推荐分配', PLAN_10MIN), ('典型错误分配', BAD_PLAN)]:
    r = audit_time_plan(plan)
    print('%s：总时长 %ds，A 段占比 %.0f%%，约 %d 字  ->  %s'
          % (label, r['total_s'], 100 * r['a_share'], r['words'],
             'OK' if (r['fits'] and r['a_ok']) else '不合格'))
    for name, t in plan:
        bar = '#' * int(t / 15)
        print('     %-14s %3ds %s' % (name, t, bar))

good = audit_time_plan(PLAN_10MIN)
bad = audit_time_plan(BAD_PLAN)
assert good['total_s'] == 600 and good['words'] == 2200
assert good['fits'] and good['a_ok'] and abs(good['a_share'] - 0.525) < 1e-9
assert not bad['a_ok'] and abs(bad['a_share'] - 120.0 / 420) < 1e-9
print('\n✅ 错误分配里 A 段只占 29% —— 面试官听完只知道你做了什么，不知道你为什么这么做。')

## ✏️ 练习 1：分格子的缺失报告

实现 `missing_report(story)`：返回一个 `OrderedDict`，键是**有缺失的格子**（'S'/'T'/'A'/'R'），
值是该格子里缺失字段名的列表（按 schema 顺序）。**完全填满的格子不出现在结果里。**

> 为什么按格子分组？因为四个格子对应四个不同的评分维度
> （问题定义 / 约束意识 / 归因能力 / 交付纪律），
> 你需要知道自己是「哪个维度」空着，而不是「哪几个字段」空着。

In [ ]:
def missing_report(story):
    # TODO: 复用 STORY_SCHEMA 与 is_filled
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
r1 = missing_report(WEAK_STORY)
assert list(r1) == ['S', 'T', 'A', 'R'], list(r1)
assert r1['S'] == ['baseline', 'metric_protocol', 'how_found'], r1['S']
assert r1['T'] == ['constraints', 'frozen'], r1['T']
assert r1['A'] == ['why', 'failed', 'validation_design'], r1['A']
assert r1['R'] == ['costs', 'verification', 'business_impact'], r1['R']
assert missing_report(TSR_STORY) == OrderedDict(), missing_report(TSR_STORY)

# 一个只缺 A 格的故事：其他维度都覆盖了，但最高区分度的一格是空的
half = {k: dict(v) for k, v in TSR_STORY.items()}
half['A'] = {'actions': TSR_STORY['A']['actions']}
r2 = missing_report(half)
assert list(r2) == ['A'] and r2['A'] == ['why', 'failed', 'validation_design']
for g, flds in r1.items():
    print('  [%s] 缺 %d 项：%s' % (g, len(flds), ', '.join(flds)))
print('✅ 练习 1 通过：弱版故事四个维度全部有缺口，A 格（归因）缺得最狠。')

## ✏️ 练习 2：涨跌表（收益与代价一起排序）

实现 `delta_table(items)`：`items` 是形如
`[{'name':..., 'before':x, 'after':y}, ...]` 的列表，
返回 `[(name, before, after, delta, pct), ...]`，其中
`delta = after - before`，`pct = delta / before`，**按 `pct` 从大到小排序**。

> 为什么要算相对变化？因为「mAP +0.035」和「小目标桶 +0.17」放在一起时，
> 绝对值会骗你 —— 相对变化才看得出哪个是主要收益。
> 代价那一侧同理：**训练时长 +40% 往往比延迟 +28% 更容易被忽略。**

In [ ]:
def delta_table(items):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
g = delta_table(TSR_STORY['R']['before_after'])
assert [x[0] for x in g] == ['<32px 桶 recall@FP=0.1', '夜间切片 recall', 'mAP50-95'], g
assert abs(g[0][3] - 0.17) < 1e-9 and abs(g[0][4] - 0.17 / 0.44) < 1e-9
assert abs(g[2][4] - 0.035 / 0.58) < 1e-9

c = delta_table(TSR_STORY['R']['costs'])
assert c[0][0] == '训练时长(GPU-h)' and abs(c[0][4] - 0.4) < 1e-9, c[0]
assert c[-1][0] == '>128px 桶 recall' and c[-1][3] < 0        # 唯一真正掉的一项
print('收益侧（按相对变化排序）：')
for n, b, a, d, p in g:
    print('   %-24s %7.3f -> %7.3f   %+.3f  (%+.1f%%)' % (n, b, a, d, 100 * p))
print('代价侧：')
for n, b, a, d, p in c:
    print('   %-24s %7.3f -> %7.3f   %+.3f  (%+.1f%%)' % (n, b, a, d, 100 * p))
print('✅ 练习 2 通过：真正最大的代价是训练时长 +40%，而不是最显眼的延迟 +28%。')

## ✏️ 练习 3：追问准备的优先级队列

实现 `priority_todo(claims)`：返回还没准备好的 `(claim_text, layer)` 列表，
**按层的优先级排序：机制 > 方差 > 口径**（同一层内保持 claims 的原顺序）。

> 优先级这么定，是因为「机制层」和「方差层」是面试官最常钻、
> 也最能区分资深度的两层；口径层则相对容易临场补。

In [ ]:
LAYER_PRIORITY = {'机制': 0, '方差': 1, '口径': 2}

def priority_todo(claims):
    # TODO: 遍历 claims x LAYERS，收集未填的，按 LAYER_PRIORITY 排序（稳定排序保持原顺序）
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
todo = priority_todo(CLAIMS)
assert len(todo) == 3, todo
assert todo == [('输入 640->960 让 <32px 桶 +0.11', '机制'),
                ('close-mosaic 最后 10 epoch 有效', '机制'),
                ('close-mosaic 最后 10 epoch 有效', '方差')], todo
assert priority_todo([CLAIMS[0]]) == []          # 第一条 claim 三层齐全
for i, (t, L) in enumerate(todo, 1):
    print('  %d. [%s] %s' % (i, L, t))
print('✅ 练习 3 通过：优先补「机制」层 —— 「什么时候这个方法会无效」是最高价值的一问。')

## ✏️ 练习 4：简历 bullet 打分器

实现 `score_bullet(text)`：按下面六项检查打分，返回 `(得分, 命中项列表)`，
命中项按 `BULLET_CHECKS` 的顺序排列。

- 五项是**关键词命中**（列表已给出）
- `'数字'` 这一项特殊：文本中出现 **>= 2 个数字**（含小数）才算命中

In [ ]:
BULLET_CHECKS = OrderedDict([
    ('动词', ['设计', '实现', '优化', '定位', '排查', '搭建', '主导', '重构']),
    ('方法', ['copy-paste', 'Mosaic', 'TIDE', 'P2 层', 'FPN', 'NMS',
              'INT8', 'QAT', 'letterbox', '分桶', 'SimOTA', 'ATSS']),
    ('数字', None),   # 特殊规则：>= 2 个数字
    ('口径', ['mAP50-95', 'mAP50', 'AP50', 'recall@', 'FP/帧', 'FP/km', 'p99']),
    ('代价', ['ms', '显存', 'GB', '延迟', 'GPU-h', '训练时长']),
    ('影响', ['预算', '检出距离', '上线', '接管', '投诉', '门禁', '反应时间']),
])
NUM_PAT = re.compile('[0-9]+(?:[.][0-9]+)?')

def score_bullet(text):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
B_WEAK = '负责交通标志检测模型的开发与优化，提升了模型精度'
B_MID  = '基于 YOLOv8 的交通标志检测，mAP50-95 从 0.58 提升到 0.615'
B_STRONG = ('设计稀有类 copy-paste + 960 分辨率方案：<32px 桶 recall@FP=0.1 从 0.44 提到 0.61，'
            'mAP50-95 0.58->0.615（3 seeds，配对 bootstrap p<0.01），'
            '端到端延迟 6.1->7.8 ms 仍在 12 ms 车端预算内')

assert score_bullet(B_WEAK)   == (1, ['动词']), score_bullet(B_WEAK)
assert score_bullet(B_MID)    == (2, ['数字', '口径']), score_bullet(B_MID)
assert score_bullet(B_STRONG) == (6, ['动词', '方法', '数字', '口径', '代价', '影响'])
for label, b in [('弱', B_WEAK), ('中', B_MID), ('强', B_STRONG)]:
    s, h = score_bullet(b)
    print('  [%s] %d/6  命中：%s' % (label, s, ', '.join(h) if h else '无'))
    print('       %s' % b[:60] + ('…' if len(b) > 60 else ''))
print('✅ 练习 4 通过：简历上每条 bullet 先过这个 6 分制，低于 4 分的一律重写。')
print('   但记住反向铁律：**写上去的每个数字，你都要能撑住三层追问。**')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def missing_report(story):
    out = OrderedDict()
    for g, fields in STORY_SCHEMA.items():
        blk = story.get(g, {})
        lack = [f for f in fields if not is_filled(blk.get(f))]
        if lack:
            out[g] = lack
    return out

In [ ]:
# 练习 2 参考答案
def delta_table(items):
    rows = []
    for it in items:
        b, a = float(it['before']), float(it['after'])
        d = a - b
        rows.append((it['name'], b, a, d, d / b))
    rows.sort(key=lambda r: r[4], reverse=True)
    return rows

In [ ]:
# 练习 3 参考答案
def priority_todo(claims):
    gaps = [(c['text'], L) for c in claims for L in LAYERS if not is_filled(c.get(L))]
    gaps.sort(key=lambda x: LAYER_PRIORITY[x[1]])     # sort 是稳定的，同层保持原顺序
    return gaps

In [ ]:
# 练习 4 参考答案
def score_bullet(text):
    hits = []
    for name, kws in BULLET_CHECKS.items():
        if name == '数字':
            if len(NUM_PAT.findall(text)) >= 2:
                hits.append(name)
        elif any(k in text for k in kws):
            hits.append(name)
    return len(hits), hits

---
## 🧪 真实工程胶囊：面试叙事准备工作台

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# 项目叙事准备工作台 —— 按顺序做，每一步都有明确的完成条件
# ══════════════════════════════════════════════════════════════════════

# ① 素材归档（T-14d）：把四个故事的原始材料捞回来
#    每个故事建一个目录，至少存下：
#      configs/        baseline 与改动后的完整配置 diff
#      metrics.json    分桶 + 分场景的完整指标（不是只有一个 mAP）
#      seeds.csv       每个种子的结果（**这是「+0.3 是不是噪声」的唯一凭据**）
#      failed.md       试过没成功的实验，每条按六段式记：
#                      假设 / 预期 / 实测 / 归因 / 可迁移结论 / 后续处理
#      latency.md      延迟口径写死：batch / 卡型 / 含不含预处理与 NMS / p50 与 p99
#    完成条件：narrative_audit() 得分 >= 90

# ② 三层追问自审（T-7d）
#    对每个 claim 写满「口径 / 方差 / 机制」三层。
#    机制层必须包含「**什么时候这个方法会无效**」——这一句是最高价值的。
#    完成条件：layer_coverage 总覆盖 >= 80%，机制层 >= 75%

# ③ 计时演练（T-2d）
#    对着秒表讲，录音回听。检查三件事：
#      · A 段是否占 >= 50%（多数人第一次只有 25-30%）
#      · 有没有说出「代价」这个词（没说 = 必被追问）
#      · 结尾有没有留钩子（「如果预算再宽 3ms，我下一步会做…」）

# ④ 简历回扫（T-2d）
#    每条 bullet 过 score_bullet()，低于 4 分重写。
#    然后反向检查：**每个写上去的数字，能不能撑住三层追问？**
#    撑不住的两个选择：补功课，或者从简历上删掉。删掉不丢人，答不上才丢人。

# ⑤ 三个版本的讲述（T-1d）
#    30 秒版 / 3 分钟版 / 10 分钟版，各练一遍。
#    最常见的失误：被问「简单说说」时讲了 8 分钟，把面试官原本想问的
#    三个话题挤掉两个。

# ⑥ 红线自检（面试当天早上，60 秒）
#    · 不虚构项目（课程实验就说课程实验，方法论完整照样加分）
#    · 不报没测过的数字（「大概三四个点吧」会让你所有精确数字一起贬值）
#    · 不泄露前雇主敏感数据（「这个具体数不方便说，我讲相对提升和方法」
#      —— 这个回答本身就是加分项）

# ⑦ 被问到不会的东西：三段式
#    ① 明确承认边界：「这个我没有实操过」
#    ② 给相邻知识：  「但我知道它要解决的问题是 X，思路上应该是 Y」
#    ③ 给行动方案：  「如果要落地，我会先做 Z 的最小验证」
#    三段齐全的「我不知道」，比含糊的假装知道得分高得多。
'''
print(RECIPE)
for token in ['seeds.csv', 'failed.md', '六段式', '什么时候这个方法会无效',
              'A 段是否占', 'score_bullet', '不虚构项目', '三段式']:
    assert token in RECIPE, token
print('✅ 工作台覆盖：素材归档 / 三层自审 / 计时演练 / 简历回扫 / 三版本 / 红线 / 不会时怎么答')

### 小结

- **面试官看不到你做过什么，只能看到你说的话。**叙事的目标不是让项目显得大，
  而是让你的**判断**显得清晰。判据是似然比：*不做也说得出来的话，说了等于没说。*
- **STAR 的检测版有 15 个必填字段**，缺一个就是一次必然的追问。
  同一个项目，弱版 16 分 / 16 条告警，完整版 100 分 / 0 条告警 ——
  差别不在项目本身，在当初有没有把 baseline、代价、失败记下来。
- **指标必须成对**：mAP↑ 配延迟/显存/训练成本，recall↑ 配同工作点 FP 率，
  新场景↑ 配旧场景不掉。*算相对变化会发现，最大的代价常常不是最显眼的那个*
  （TSR 样例里训练时长 +40% > 延迟 +28%）。
- **失败实验是资深度的信号**，但必须有归因。四种类型（假设错 / 实现有 bug /
  代价不可接受 / 不可复现）各自传达不同的能力信号；
  「保留为天花板参考」是把失败转化为信息的标准用法。
  *反例是「试了很多都不行」—— 没有归因的失败消除的不确定性接近零。*
- **三层追问自审**（口径 / 方差 / 机制）是准备追问的机械方法。
  多数人在方差层和机制层崩，而「什么时候这个方法会无效」是最高价值的一问。
- **A 段必须占 10 分钟里的一半以上**。背景好讲、判断难讲，所以要靠秒表强制。

下一站：**模块 05 · 面试演练题库与满分答案骨架** —— 白板题的参考实现、
每题的评分要点与追问预案，以及反问环节该问什么。